In [0]:
from pyspark.sql.functions import *

In [0]:
%run /Workspace/Repos/projects/databricks-batch-pipeline/modules/bronze_layer

In [0]:
# from modules.bronze_layer import BronzeLayer

In [0]:
pipeline_name = "hr"
conf = (
    spark.table("session_11_framework.config_table")
    .filter(col("pipeline_name") == pipeline_name)
    .select(
        "header", 
        "delimiter", 
        "multiline", 
        "escape_option", 
        "quote_option", 
        "table_name", 
        "file_path")
    ).first()
conf

In [0]:
csv_options = {
    "delimiter": conf.delimiter,
    "header": conf.header,
    "multiline": conf.multiline,
    "escape_option": conf.escape_option,
    "quote_option": conf.quote_option
}

# 2. เตรียม Path และชื่อตาราง
RAW_FILE_PATH = "/Workspace/Repos/projects/databricks-batch-pipeline/data/HRDataset_v14.csv"
TABLE_NAME = "hr"

In [0]:
bronze_pipeline = BronzeLayer(
    file_path=conf.file_path,
    table_name=conf.table_name,
    write_mode="overwrite",
    format_data="delta",
    encrypt_col= ["EmpID", "Salary", "ManagerID"],
    reader_options=csv_options
)


In [0]:
print(f"⏳ กำลังอ่านข้อมูลจาก: {RAW_FILE_PATH}")
df_bronze = bronze_pipeline.read_from_raw()

# (Optional) ลองพรีวิวข้อมูลดูว่า Column Audit ถูกสร้างขึ้นมาถูกต้องไหม
display(df_bronze.limit(5))



In [0]:
encrypt_key = dbutils.secrets.get("hr", "encrypt_key")
df = bronze_pipeline.encrypt_df(df_bronze, encrypt_key)

In [0]:
df.display()

In [0]:
# 3. สั่งให้เขียนข้อมูลลง Bronze Table

bronze_pipeline.write_file(df)

In [0]:
display(df)